# SWAN-SF Dataset EDA.

### Link: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/EBCFKM



## Download Dataset to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import requests
from google.colab import drive


# 2. DIRECTORY CREATION
data_dir = '/content/drive/MyDrive/solar_flare_forecasting/Data'
os.makedirs(data_dir, exist_ok=True)

# 3. DATASET DICTIONARY
datasets = {
    "addenda.tar.gz": "https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/EBCFKM/VI66WW",
    "partition1_instances.tar.gz": "https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/EBCFKM/BMXYCB",
    "partition2_instances.tar.gz": "https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/EBCFKM/TCRPUD",
    "partition3_instances.tar.gz": "https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/EBCFKM/PTPGQT",
    "partition4_instances.tar.gz": "https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/EBCFKM/FIFLFU",
    "partition5_instances.tar.gz": "https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/EBCFKM/QC2C3X",
    "SWAN.tar.gz": "https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/EBCFKM/K9AOSI"
}

# 4. EFFICIENT DOWNLOAD
def download_files(dataset_dict, target_folder):
    for filename, url in dataset_dict.items():
        file_path = os.path.join(target_folder, filename)

        if os.path.exists(file_path):
            print(f"[SKIP] {filename} already exists.")
            continue

        print(f"[DOWNLOADING] {filename}...")
        try:
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                with open(file_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            print(f"[COMPLETE] Finished downloading {filename}.")
        except Exception as e:
            print(f"[ERROR] Failed to download {filename}: {e}")

download_files(datasets, data_dir)

# 5. EXTRACTION (OPTIONAL)
'''
print("\n--- Starting High-Speed Local Extraction ---")
# Using native Linux tar tool via system command for maximum speed
for file_name in datasets.keys():
    archive_path = os.path.join(drive_source_folder, file_name)

    if os.path.exists(archive_path):
        print(f"⚡ Unpacking {file_name} into Colab SSD...")
        # -xf: extract file, -C: target directory
        os.system(f"tar -xf {archive_path} -C {local_extract_folder}")
    else:
        print(f"⚠️ Could not find archive for {file_name}, skipping extraction.")

print("\n🎉 All processes complete! Data is ready in '/content/solar_flare_data'.")
'''

Extracting partition 1

In [ ]:
import os

# Where the archive currently lives
drive_data_dir = '/content/drive/MyDrive/solar_flare_forecasting/Data'
# True high-speed local Colab SSD path
local_extract_dir = '/content/solar_flare_data'

# Ensure the local extraction directory exists
os.makedirs(local_extract_dir, exist_ok=True)

files_to_extract = {
    "partition1_instances.tar.gz": "https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/EBCFKM/BMXYCB"
}

print("\n--- Starting High-Speed Local Extraction ---")

for file_name in files_to_extract.keys():
    archive_path = os.path.join(drive_data_dir, file_name)

    if os.path.exists(archive_path):
        print(f"⚡ Unpacking {file_name} into Colab SSD ({local_extract_dir})...")
        # -xzf: extract gzipped file, -C: target local directory
        # Using native system tar is indeed the fastest method
        exit_code = os.system(f"tar -xzf {archive_path} -C {local_extract_dir}")

        if exit_code == 0:
            print(f"✅ Successfully unpacked {file_name}!")
        else:
            print(f"❌ Error occurred while unpacking {file_name}. Exit code: {exit_code}")
    else:
        print(f"⚠️ Could not find archive at {archive_path}, skipping extraction.")

print(f"\n🎉 All processes complete! Data is ready in '{local_extract_dir}'.")

## Explore Single file

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### 1. TOP-LEVEL DIRECTORY STRUCTURE

In [ ]:
print("=" * 60)
print("TOP-LEVEL STRUCTURE")
print("=" * 60)

for root, dirs, files in os.walk(local_extract_dir):
    depth = root.replace(local_extract_dir, '').count(os.sep)
    indent = '  ' * depth
    print(f"{indent}{os.path.basename(root)}/")
    if depth < 2:  # only show files for top 2 levels
        for f in files[:5]:  # show first 5 files
            print(f"{indent}  └─ {f}")
        if len(files) > 5:
            print(f"{indent}  └─ ... and {len(files) - 5} more files")

### 2. COUNT FILES IN FL AND NF

In [ ]:

print("\n" + "=" * 60)
print("FILE COUNTS")
print("=" * 60)

# Adjust these paths based on what you see above
fl_dir = os.path.join(local_extract_dir, 'FL')
nf_dir = os.path.join(local_extract_dir, 'NF')

# Search one level deeper if needed
if not os.path.exists(fl_dir):
    for root, dirs, files in os.walk(local_extract_dir):
        if 'FL' in dirs:
            fl_dir = os.path.join(root, 'FL')
            nf_dir = os.path.join(root, 'NF')
            break

fl_files = sorted(os.listdir(fl_dir))
nf_files = sorted(os.listdir(nf_dir))

print(f"FL (Flaring)     : {len(fl_files):,} files")
print(f"NF (Non-Flaring) : {len(nf_files):,} files")
print(f"Total            : {len(fl_files) + len(nf_files):,} files")
print(f"\nClass imbalance ratio  →  1 : {len(nf_files)/len(fl_files):.1f}  (FL : NF)")

### 3. INSPECT A SINGLE FILE

In [ ]:
print("\n" + "=" * 60)
print("SINGLE FILE INSPECTION")
print("=" * 60)

sample_fl_path = os.path.join(fl_dir, fl_files[0])
print(f"File: {fl_files[0]}")
print(f"Size: {os.path.getsize(sample_fl_path) / 1024:.2f} KB\n")

# Fix: use tab separator
df_sample = pd.read_csv(sample_fl_path, sep='\t')

print("Shape:", df_sample.shape)
print(f"  → {df_sample.shape[0]} time steps  x  {df_sample.shape[1]} columns\n")
print("Columns:")
for i, col in enumerate(df_sample.columns):
    print(f"  [{i:2d}] {col}")

print("\nFirst 3 rows:")
print(df_sample.head(3))

print("\nData types:")
print(df_sample.dtypes)

print("\nMissing values per column:")
missing = df_sample.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None")

Why are certain label columns missing?

#### CHECK _LABEL COLUMNS ACROSS PARTITIONS

In [ ]:
import os
import pandas as pd

local_extract_dir = '/content/solar_flare_data'

# Define partition paths
partitions = {
    'partition1': os.path.join(local_extract_dir, 'partition1'),
}

# Initialize a counter for the number of files read
files_read_count = 0

for part_name, part_path in partitions.items():
    if not os.path.exists(part_path):
        print(f"⚠️  {part_name} not found at {part_path}, skipping.")
        continue

    fl_dir = os.path.join(part_path, 'FL')
    sample_file = sorted(os.listdir(fl_dir))[0]

    # Increment counter
    df = pd.read_csv(os.path.join(fl_dir, sample_file), sep='\t')
    files_read_count += 1

    print(f"\n{'='*60}")
    print(f"{part_name} — {sample_file[:50]}")
    print(f"{'='*60}")

    label_cols = [c for c in df.columns if 'LABEL' in c]
    for col in label_cols:
        non_null = df[col].notna().sum()
        sample_vals = df[col].dropna().unique()[:3]
        print(f"  {col:<25} | non-null: {non_null:2d}/60 | sample: {sample_vals}")

print(f"\nTotal CSV files read in this cell: {files_read_count}")

In [ ]:
# ─────────────────────────────────────────
# READ ALL FILES AND CHECK LABEL COMPLETENESS
# ─────────────────────────────────────────
import os
import pandas as pd

fl_dir = '/content/solar_flare_data/partition1/FL'
nf_dir = '/content/solar_flare_data/partition1/NF'

# These were the columns found to be 100% null in the first sample file
blank_cols_target = [
    'BFLARE_LABEL', 'CFLARE_LABEL', 'MFLARE_LABEL', 'XFLARE_LABEL',
    'BFLARE_LABEL_LOC', 'CFLARE_LABEL_LOC', 'MFLARE_LABEL_LOC', 'XFLARE_LABEL_LOC'
]

# ── Load and Check FL files ──
fl_records = []
fl_with_data_count = 0

if os.path.exists(fl_dir):
    for fname in sorted(os.listdir(fl_dir)):
        fpath = os.path.join(fl_dir, fname)
        df = pd.read_csv(fpath, sep='\t')

        # Check if ANY of the target label columns have ANY non-null data
        if df[blank_cols_target].notna().any().any():
            fl_with_data_count += 1

        df['_filename'] = fname
        df['_label']    = 1
        fl_records.append(df)

# ── Load and Check NF files ──
nf_records = []
nf_with_data_count = 0

if os.path.exists(nf_dir):
    # Limiting to 1000 for RAM safety as established previously
    for fname in sorted(os.listdir(nf_dir))[:1000]:
        fpath = os.path.join(nf_dir, fname)
        df = pd.read_csv(fpath, sep='\t')

        if df[blank_cols_target].notna().any().any():
            nf_with_data_count += 1

        df['_filename'] = fname
        df['_label']    = 0
        nf_records.append(df)

# ── Combine and Report ──
if fl_records and nf_records:
    df_fl_all = pd.concat(fl_records, ignore_index=True)
    df_nf_all = pd.concat(nf_records, ignore_index=True)

    print(f"Summary of files with actual labels in target columns:")
    print(f"FL Files with data: {fl_with_data_count} / {len(fl_records)}")
    print(f"NF Files with data: {nf_with_data_count} / {len(nf_records)}")

    print(f"\nTotal instances loaded: {len(fl_records) + len(nf_records):,}")
    print(f"FL shape: {df_fl_all.shape}")
    print(f"NF shape: {df_nf_all.shape}")
else:
    print("❌ Records not found.")

### 5. COMPARE FL vs NF SAMPLE

In [ ]:
df_fl = pd.read_csv(os.path.join(fl_dir, fl_files[0]), sep='\t')
df_nf = pd.read_csv(os.path.join(nf_dir, nf_files[0]), sep='\t')

print(f"FL sample shape: {df_fl.shape}")
print(f"NF sample shape: {df_nf.shape}")

print("\nChecking time series lengths across 10 FL files...")
fl_lengths = [
    pd.read_csv(os.path.join(fl_dir, f), sep='\t').shape[0]
    for f in fl_files[:10]
]
print(f"  Lengths: {fl_lengths}")
print(f"  All same length: {len(set(fl_lengths)) == 1}")